In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os

load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

llm = ChatGroq(model="qwen/qwen3-32b")


In [8]:
from pydantic import BaseModel, Field
from typing import List

class Section(BaseModel):
    sub_heading: str = Field(description="A short sub-heading for the section")
    body: str = Field(description="The main content of the section")

class Sections(BaseModel):
    sections: List[Section] = Field(description="A list of sections")

blog_planner = llm.with_structured_output(Sections)

In [15]:
from typing import TypedDict, Annotated
import operator

class State(TypedDict):
    title: str
    sections: Sections
    completed_sections: Annotated[list, operator.add]
    final_blog_post: str

class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]

In [46]:
from langchain.messages import HumanMessage, SystemMessage
from langgraph.types import Send

def blog_post_planner(state: State):
    """ Plan the sections of the blog post"""
    blog_sections = blog_planner.invoke(
        [
            SystemMessage(content="You are a blog post planner"),
            HumanMessage(content="Plan a blog post about " + state["title"])
        ]
    )
    return {"sections": blog_sections}

def blog_section_writer(state: WorkerState):
    """ Write a section of the blog post"""
    print(f"Writing section: {state['section'].sub_heading}")
    section = llm.invoke(
        [
            SystemMessage(content="You are a blog post writer. Include no preamble for each section. Use markdown formatting. Only include the section content."),
            HumanMessage(content=f"Write a section of the blog post with sub-heading: {state["section"].sub_heading} and body: {state["section"].body}. Don't Add thinking messages. Only include the section content. " )
        ]
    )
    return {"completed_sections": [section.content]}


def blog_post_writer(state: State):
    """ Write the blog post"""
    final_blog_post = '\n\n---\n\n'.join(state["completed_sections"])
    return {"final_blog_post": final_blog_post}

def blog_section_scheduler(state: State):
    """ Schedule the sections of the blog post to each worker"""
    return [Send("writer", {"section": s}) for s in state["sections"].sections]


In [44]:
from langgraph.graph import START, END, StateGraph

blog_builder = StateGraph(State)

blog_builder.add_node("planner", blog_post_planner)
blog_builder.add_node("writer", blog_section_writer)
blog_builder.add_node("synthezier", blog_post_writer)

blog_builder.add_edge(START, "planner")
blog_builder.add_conditional_edges("planner", blog_section_scheduler, ["writer"])
blog_builder.add_edge("writer", "synthezier")
blog_builder.add_edge("synthezier", END)


blogger = blog_builder.compile()

In [45]:
state = blogger.invoke({"title": "My Blog post on TajMahal"})
print(state)

from IPython.display import Markdown
Markdown(state["final_blog_post"])

Writing section: Introduction to the Taj Mahal
Writing section: Historical Background
Writing section: Architectural Marvel
Writing section: Cultural and Symbolic Significance
Writing section: Visiting the Taj Mahal
Writing section: Conservation Efforts
Writing section: Conclusion
{'title': 'My Blog post on TajMahal', 'sections': Sections(sections=[Section(sub_heading='Introduction to the Taj Mahal', body='The Taj Mahal, a UNESCO World Heritage Site, is one of the most iconic symbols of love and architectural brilliance. Located in Agra, India, it was commissioned by Emperor Shah Jahan in memory of his beloved wife Mumtaz Mahal. This section will introduce its historical significance and global recognition.'), Section(sub_heading='Historical Background', body="Built between 1632 and 1648, the Taj Mahal's construction involved thousands of artisans and laborers. Learn about the tragic love story of Shah Jahan and Mumtaz Mahal, the Mughal era's influence, and the challenges faced during 

<think>

</think>

# Introduction to the Taj Mahal

The Taj Mahal, a UNESCO World Heritage Site, is one of the most iconic symbols of love and architectural brilliance. Located in Agra, India, it was commissioned by Emperor Shah Jahan in memory of his beloved wife Mumtaz Mahal. This section will introduce its historical significance and global recognition.

---

<think>
Okay, let me start by understanding the user's request. They want a section of a blog post with the subheading "Historical Background" and specific content about the Taj Mahal's construction, the love story of Shah Jahan and Mumtaz Mahal, the Mughal era's influence, and the challenges during its creation. The user provided a sample response, so I need to make sure mine is different but covers the same points.

First, the Taj Mahal was built between 1632 and 1648. I should mention the number of workers involved—thousands of artisans and laborers. Next, the tragic love story: Shah Jahan's wife Mumtaz Mahal died during childbirth, leading him to commission the Taj Mahal as a mausoleum. The Mughal era's influence is important—maybe touch on the architectural style, blending Persian, Islamic, and Indian elements. Challenges during construction could include the scale of the project, the logistics of moving materials, the use of specific materials like white marble, and the precision required in the design. Also, the labor system under Akbar might be relevant. Need to avoid any markdown and ensure it's in the user's desired format without thinking steps. Let me structure this into paragraphs, each focusing on a key point. Make sure it's concise and flows well, using the keywords provided. Check for any historical inaccuracies. For example, confirming the dates, the role of Ustad Ahmad Lahori as chief architect, and the cost implications. Avoid repeating the sample response's structure but use the same information points. Double-check the word count to ensure it's comprehensive but not too lengthy. Alright, time to put it all together in a coherent section.
</think>

### Historical Background  
Erected between 1632 and 1648 under the patronage of Mughal Emperor Shah Jahan, the Taj Mahal stands as a testament to enduring love and imperial grandeur. This architectural marvel was commissioned in memory of Shah Jahan’s beloved wife, Mumtaz Mahal, who tragically died during the birth of their 14th child in 1631. Her passing left the emperor grief-stricken, prompting him to conceive the Taj Mahal as a celestial mausoleum to house her remains and honor their union.  

The Mughal Empire’s zenith during the 17th century provided the cultural and economic foundation for such an ambitious project. Drawing from Persian, Islamic, and Indian architectural traditions, the Taj Mahal reflects the Mughal ethos of blending artistry with power. Its symmetrical design, intricate pietra dura inlays, and towering central dome exemplify the era’s mastery of craftsmanship.  

Construction involved an estimated 20,000 artisans and laborers sourced from across the empire and beyond, including skilled stonemasons from Persia, calligraphers from Turkey, and metalworkers from Central Asia. The project faced logistical challenges, such as transporting 28 types of semi-precious stones and white marble from distant quarries. Despite these hurdles, the Taj Mahal was completed in 1653, a labor of devotion that solidified its legacy as one of history’s most iconic monuments.

---

<think>
Okay, the user wants me to write a section for a blog post with the subheading "Architectural Marvel" and specific content about the Taj Mahal's design. Let me start by recalling the key elements they mentioned: Mughal, Persian, Islamic architecture. I need to discuss the central dome, minarets, inlay work, and symmetrical gardens.

First, the central dome is iconic. I should mention its shape, maybe compare it to a lotus. Then the minarets – four of them, their purpose, and how they frame the dome. The inlay work with semi-precious stones is important too; perhaps mention the floral patterns and the technique used. Symmetrical gardens, maybe talk about the Charbagh layout, the reflecting pools, and how they complement the structure. Also, the blend of architectural styles should be highlighted to show the synthesis of different influences. Need to ensure the language is engaging but factual. Avoid technical jargon so it's accessible. Check for any historical inaccuracies. Maybe add terms like "iwan" or "pishtaq" if appropriate. Make sure each feature is clearly explained and connected to the overall marvel aspect. Keep paragraphs concise. Let me organize the points in a logical flow: start with the central dome, then minarets, inlay work, gardens, and conclude with the synthesis of styles. That should cover the user's requirements without missing anything. Let me put it all together now.
</think>

**Architectural Marvel**  
The Taj Mahal stands as a testament to the zenith of Mughal architecture, seamlessly merging Persian grandeur, Islamic geometry, and Indian craftsmanship. At its heart is the central dome, an inverted lotus in form, rising 35 meters above the marble base. This bulbous dome, flanked by four smaller domes, creates a celestial silhouette that has captivated visitors for centuries. Four slender minarets, each 40 meters tall, frame the structure symmetrically, their tapering design ensuring stability and visual harmony.  

The inlay work, a hallmark of Mughal artistry, adorns the white Makrana marble with intricate patterns of floral motifs and arabesques. Semi-precious stones like lapis lazuli, jade, and carnelian are meticulously embedded to create a mosaic of color that shifts with the light. The symmetry of the complex extends to its gardens, designed in the Persian Charbagh layout, where reflecting pools and cypress trees mirror the monument’s elegance. Every element—from the iwan (arched portal) to the pishtaq (ornate gateway)—embodies a mastery of proportion, balance, and spiritual symbolism, making the Taj Mahal not just a mausoleum, but a timeless ode to love and architectural ingenuity.

---

<think>
Okay, I need to write a section for a blog post about the Taj Mahal's cultural and symbolic significance. The user wants me to focus on how it represents eternal love and cultural heritage, and also mention its role in literature, art, and global pop culture as a symbol of devotion.

First, I should start by explaining the basic symbolism of the Taj Mahal. It's a mausoleum built by Shah Jahan for Mumtaz Mahal, so the love story is central. I should mention the historical context here, maybe the materials used and the craftsmanship to emphasize the cultural heritage aspect.

Next, I need to connect it to literature and art. Think of famous poems or books that reference the Taj Mahal. Maybe mention some Indian literature where it's a metaphor for love. Also, in art, perhaps how it's been depicted in paintings or films. For global pop culture, maybe reference movies, songs, or other media that feature the Taj Mahal as a symbol. Examples could include Bollywood films, Western movies, or even video games and music videos.

I should also highlight how it's more than just a monument—it's a cultural icon that represents devotion across different mediums. Make sure to tie everything back to the themes of eternal love and heritage. Avoid adding any personal opinions, just stick to the facts and examples provided.

Need to check if there are any specific works or references that are commonly cited. For example, the Taj Mahal has been in films like "The Taj Mahal" (1963) or "Bobby" (1973). In Western media, maybe "The Mummy" (1999) includes it. Also, artists like Leonardo da Vinci were inspired by it in their designs? Wait, is that true? Maybe better to avoid that if unsure. Focus on more verifiable examples.

Also, mention its UNESCO status as a World Heritage Site to emphasize cultural heritage. Maybe touch on how it's a blend of Mughal, Persian, Islamic, and Indian architectural styles, showing its rich cultural tapestry.

Make sure the section flows smoothly, starting with the historical significance, then moving into literature, art, and pop culture. Conclude by reinforcing its enduring legacy as a symbol of love and heritage.
</think>

**Cultural and Symbolic Significance**  
The Taj Mahal, a 17th-century Mughal mausoleum, stands as a timeless testament to eternal love and cultural fusion. Commissioned by Emperor Shah Jahan in memory of his wife Mumtaz Mahal, its intricate marble inlays, symmetrical gardens, and domed architecture reflect both Islamic and Indian artistic traditions. The monument transcends its role as a royal tomb, embodying the profound connection between devotion and artistry.  

In literature, the Taj Mahal has inspired countless poets and writers. Urdu and Persian poets have long depicted it as the pinnacle of romantic sacrifice, while modern Indian authors, such as Khushwant Singh, use it as a metaphor for unrequited love and historical legacy. Its imagery appears in global fiction, symbolizing idealized devotion in novels like *The Shadow of the Banyan* and *Midnight’s Children*.  

Art and cinema have further cemented its global iconography. From miniature paintings of Mughal courts to contemporary photography, the Taj Mahal’s ethereal beauty has captivated artists. In film, it has served as a backdrop for love stories in Bollywood classics like *Bobby* and *Dilwale Dulhania Le Jayenge*, as well as Western narratives such as *The Mummy* and *Slumdog Millionaire*. Its presence in music, fashion, and digital media—from Adele’s music video to viral social media trends—ensures its symbolism remains relevant across generations.  

As a UNESCO World Heritage Site, the Taj Mahal not only preserves Mughal heritage but also bridges cultures, inviting the world to reflect on love’s enduring power. Its legacy endures as a universal emblem of devotion, transcending borders and time.

---

<think>
</think>

### Visiting the Taj Mahal

For travelers planning a visit to the Taj Mahal, timing is everything. The best times to visit are during the early morning or late afternoon to avoid the midday heat and crowds. Sunrise offers a magical experience as the marble structure bathes in golden light, while sunset creates a soft, romantic glow. Both times provide excellent photo opportunities with fewer people around.

Entry fees vary depending on nationality and whether you're a solo traveler or part of a group. Indians pay significantly less than international visitors, and there are discounts for students and groups. Photography is allowed, but the use of tripods and professional equipment is restricted. Drone photography is strictly prohibited.

Near the Taj Mahal, travelers should also consider visiting Agra Fort, a UNESCO World Heritage Site with stunning Mughal architecture, and Fatehpur Sikri, an abandoned city that was once the capital of the Mughal Empire. Both destinations are within a short drive from Agra and offer fascinating insights into India's rich history.

---

<think>
</think>

**Conservation Efforts**

Preserving the Taj Mahal is a complex task due to threats like pollution and the heavy footfall of tourists. Air pollution, primarily from nearby industries and vehicular emissions, has caused discoloration of the marble, while water pollution in the Yamuna River poses a risk to the structure's foundation. The sheer number of visitors each year also contributes to wear and tear on the monument.

To counter these challenges, the Indian government has launched several initiatives. The Taj Trapezium Zone (TTZ), established in 1996, aims to reduce pollution in the surrounding area by regulating industries and promoting cleaner technologies. Additionally, the Archaeological Survey of India (ASI) regularly conducts restoration work and employs advanced cleaning techniques to preserve the marble's pristine appearance.

Non-governmental organizations and environmental groups have also played a crucial role. NGOs like the Centre for Science and Environment advocate for stricter pollution controls and raise public awareness about sustainable tourism. Their efforts have contributed to more responsible tourism policies, including timed entry slots and visitor caps.

Through a combination of government policies, environmental advocacy, and community involvement, the Taj Mahal continues to be protected for future generations.

---

<think>
Okay, the user wants me to write a conclusion section for a blog post about the Taj Mahal. Let me start by understanding the requirements. The subheading is "Conclusion" and the body needs to summarize the Taj Mahal's enduring legacy as a masterpiece of human creativity and a testament to love. Also, I need to encourage readers to experience it firsthand while respecting its historical value. 

First, I should highlight the Taj Mahal's status as a masterpiece. Mention its architectural brilliance and the story of love behind it. Maybe use phrases like "timeless symbol of love" and "architectural marvel." 

Next, I need to connect it to human creativity. Perhaps talk about the craftsmanship, the materials used, and the dedication involved in its construction. Words like "testament to human ingenuity" could work here.

Then, the part about encouraging visits. I should emphasize experiencing its grandeur in person but also remind them to respect its historical significance. Phrases like "step into its shadow" and "cherish its history" might be good. Also, mention preserving it for future generations to tie in the importance of conservation.

Avoid any markdown formatting, just plain text. The user specified not to include thinking messages, so I need to stick strictly to the section content. Let me structure the conclusion with a strong opening sentence, followed by supporting points, and a closing call to action. Make sure it's concise and flows well, keeping paragraphs short for readability. Check for any historical inaccuracies, but since it's a blog post, some poetic language is acceptable. Alright, let me put it all together now.
</think>

**Conclusion**  
The Taj Mahal stands as an eternal symbol of love and a pinnacle of architectural brilliance, transcending time and culture. Its intricate marble inlays, symmetrical grandeur, and poetic symmetry reflect the heights of human creativity and devotion. More than a monument, it is a narrative of passion, loss, and the enduring power of art to immortalize emotion. As one of the world’s most iconic landmarks, it invites visitors to witness the fusion of history, artistry, and romance. To experience the Taj Mahal is to step into a living testament of humanity’s capacity for beauty and connection. Let its legacy inspire awe while reminding us to cherish and protect the treasures of our shared heritage. Plan your visit, but tread with reverence—for in its stones lies a story worth safeguarding for generations to come.